# TrafficGuard · Randomised Smoothing — Certified Defence
**COMP47250 · Team Software Project · Project P14 · UCD Summer 2026**

**Owner:** Rian (Defence track) &nbsp;|&nbsp; **Contributor:** Payal (Attack track)

---

### What this notebook does

Implements **Randomised Smoothing** (Cohen et al., ICML 2019) — the only defence
in this project that provides a **provable, mathematical guarantee**: no adversarial
attack with L₂ perturbation smaller than the certified radius R can flip the prediction.

Every other defence just *reduces* attack effectiveness empirically. This one *proves*
it for a given radius — regardless of how sophisticated the attack is.

---

### How it works

The core idea: instead of predicting on the raw image x, predict on many
**noise-corrupted copies** of x and take the majority vote.

```
Smoothed classifier g(x) = argmax_c  P[f(x + ε) = c]   where ε ~ N(0, σ²I)
```

The certified radius R is computed from the majority-vote probability p_A:

```
R = σ · Φ⁻¹(p_A)
```

Where Φ⁻¹ is the inverse Gaussian CDF and p_A is a **statistical lower bound** on
the probability of the top class — computed via Clopper-Pearson confidence interval
over N Monte Carlo samples (Cohen et al. 2019, Appendix C).

---

### The key metric this unlocks

Certified Accuracy at radius R = fraction of test images that are:
  1. Correctly classified by the smoothed classifier AND
  2. Provably robust — certified radius ≥ R

This is a strictly stronger claim than empirical robust accuracy.

---

### Reference
> Cohen, Rosenfeld & Kolter. *Certified Adversarial Robustness via Randomized Smoothing.*
> ICML 2019. https://arxiv.org/abs/1902.02918

---

### Notebook Structure

| # | Section |
|---|---|
| 0 | Setup & Imports |
| 1 | Configuration |
| 2 | Load the Trained ResNet18 |
| 3 | The Smoothed Classifier |
| 4 | Prediction with Abstention |
| 5 | Certified Radius Computation |
| 6 | Evaluate on Test Set |
| 7 | Certified Accuracy Curve |
| 8 | Sigma Sensitivity Analysis |
| 9 | Visualise — Certified vs Uncertified |
| 10 | Summary |

---
## 0. Setup & Imports

In [ ]:
!pip install statsmodels --quiet   # for Clopper-Pearson lower bound

In [ ]:
import os, math, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm as scipy_norm
from statsmodels.stats.proportion import proportion_confint
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
print('All imports OK.')

---
## 1. Configuration

### Key hyperparameters

| Parameter | Symbol | Default | Effect |
|---|---|---|---|
| `SIGMA` | σ | 0.25 | Noise std dev. Larger = bigger certified radius but lower accuracy |
| `N` | N | 1000 | Monte Carlo samples for certification. Larger = tighter bound, slower |
| `N0` | N₀ | 100 | Samples for prediction-only step. Smaller is fine — just used to pick the class |
| `ALPHA` | α | 0.001 | Failure probability. 0.001 = 99.9% confidence the certificate holds |

> **Sigma tradeoff:** σ = 0.12 → small radius, high accuracy. σ = 0.50 → large radius,
> lower accuracy. σ = 0.25 is the standard starting point from the original paper.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = Path(r'checkpoints/best.pt')
MANIFEST_PATH   = Path(r'data/processed/labelled_manifest.csv')
IMAGES_DIR      = Path(r'data/raw/MIO-TCD-Localization/train')

# ── Class mapping — must match Soham's training notebook exactly ──────────────
CLASS_NAMES  = ['Low', 'Medium', 'High']
LABEL_MAP    = {'Low': 0, 'Medium': 1, 'High': 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_CLASSES  = 3

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Randomised smoothing hyperparameters ──────────────────────────────────────
SIGMA = 0.25    # noise standard deviation — start here, sweep in Section 8
N0    = 100     # samples for the initial prediction pass (cheap)
N     = 1000    # samples for the certification pass (expensive — increase to 10000 for final numbers)
ALPHA = 0.001   # statistical failure probability (0.001 = 99.9% confidence)

# ── Evaluation ────────────────────────────────────────────────────────────────
N_EVAL = 100    # number of test images to certify
                # each image takes ~N forward passes — 100 images × 1000 samples = 100K passes
                # on GPU this takes ~2-5 min. Increase to 500 for final report numbers.

ABSTAIN = -1    # sentinel value returned when the smoothed classifier abstains

print(f'Checkpoint  : {CHECKPOINT_PATH}')
print(f'Sigma (σ)   : {SIGMA}')
print(f'N₀          : {N0}  (prediction samples)')
print(f'N           : {N}  (certification samples)')
print(f'Alpha (α)   : {ALPHA}  (failure probability)')
print(f'N_EVAL      : {N_EVAL}  (test images to certify)')

---
## 2. Load the Trained ResNet18

Exact same architecture and loading pattern as Soham's training notebook.
The base classifier f is what we'll wrap with randomised smoothing.

In [ ]:
def build_resnet18(num_classes=NUM_CLASSES):
    """
    Rebuild the EXACT TrafficGuard architecture from trafficguard_model_v1.ipynb.
    Must match Soham's notebook exactly — same Dropout(0.3) + Linear(512→3).
    """
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes),
    )
    return model


def load_base_classifier(checkpoint_path, device):
    """
    Load the trained ResNet18 (the 'base classifier' f in the RS literature).
    Returns in eval() mode — Dropout must be OFF for deterministic MC sampling.
    """
    if not Path(checkpoint_path).exists():
        raise FileNotFoundError(
            f'Checkpoint not found: {checkpoint_path}\n'
            f'Make sure Soham has committed checkpoints/best.pt to the repo.'
        )

    model      = build_resnet18().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    model.eval()   # CRITICAL: eval() disables Dropout for reproducible MC samples

    epoch   = checkpoint.get('epoch', '?')
    val_acc = checkpoint.get('val_acc', None)
    print(f'[Base classifier] Loaded: epoch {epoch}' +
          (f', val_acc = {val_acc:.2%}' if val_acc else ''))
    return model


base_classifier = load_base_classifier(CHECKPOINT_PATH, DEVICE)
print(f'[Base classifier] Device: {DEVICE}')

---
## 3. The Smoothed Classifier

The smoothed classifier g wraps the base classifier f.
For every query x, it:
1. Generates N copies of x, each corrupted with independent Gaussian noise ε ~ N(0, σ²I)
2. Runs each noisy copy through the base classifier f
3. Returns the majority vote (the class predicted most often)

The key insight from Cohen et al.: this majority-vote classifier g is **certifiably robust**
within an L₂ ball of radius R = σ · Φ⁻¹(p_A), where p_A is the probability that
the majority class wins — because an adversary can't change the majority vote by moving
x a distance less than R in L₂ space.

### 3.1 Inference transform

Important: we use **ToTensor only, no ImageNet normalisation** here.
The reason: Gaussian noise is added in [0,1] pixel space *before* the model.
Normalisation happens inside the model (same setup as ml.py's NormalizedResNet).
If we normalised first and then added noise in normalised space, the noise level
would be implicitly rescaled channel-by-channel, making σ inconsistent across
channels — the certified radius formula would then be technically incorrect.

In [ ]:
# ── Inference transform: ToTensor only — NO ImageNet normalisation ─────────────
# Normalisation happens inside the model (via NormalizedResNet or the checkpoint's
# internal normalisation). Noise is added in [0,1] pixel space.
_TO_PIXELS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),    # [0, 1], shape (3, 224, 224)
])


def _pil_to_pixels(image: Image.Image) -> torch.Tensor:
    """PIL → (1, 3, 224, 224) float tensor in [0, 1]."""
    if image.mode != 'RGB':
        image = image.convert('RGB')
    return _TO_PIXELS(image).unsqueeze(0).to(DEVICE)


def _predict_noisy_batch(x: torch.Tensor, n: int, sigma: float) -> torch.Tensor:
    """
    Generate n noisy copies of x, run each through the base classifier,
    return a count vector of shape (num_classes,) — how many times each class won.

    Args:
        x     : (1, 3, 224, 224) pixel tensor in [0, 1]
        n     : number of Monte Carlo samples
        sigma : Gaussian noise standard deviation

    Why batch_size=64 inner loop:
        Generating n=1000 samples simultaneously as one batch of size 1000 would
        require 1000 × (3 × 224 × 224) × 4 bytes ≈ 600MB GPU memory.
        Chunking into batches of 64 limits peak usage to ~40MB per chunk.
    """
    counts     = torch.zeros(NUM_CLASSES, dtype=torch.long)
    batch_size = 64
    remaining  = n

    with torch.no_grad():
        while remaining > 0:
            this_batch = min(batch_size, remaining)
            # Add Gaussian noise in pixel space, clip to [0, 1]
            noise   = torch.randn(this_batch, *x.shape[1:], device=DEVICE) * sigma
            x_noisy = (x.expand(this_batch, -1, -1, -1) + noise).clamp(0, 1)

            # For NormalizedResNet (from ml.py), normalisation is applied inside
            # forward(). If using a plain ResNet18, apply normalisation here:
            # x_noisy = transforms.functional.normalize(x_noisy, IMAGENET_MEAN, IMAGENET_STD)

            logits  = base_classifier(x_noisy)
            preds   = logits.argmax(dim=1).cpu()
            for p in preds:
                counts[p] += 1
            remaining -= this_batch

    return counts   # (num_classes,) count of votes per class


print('_predict_noisy_batch() defined.')

---
## 4. Prediction with Abstention (PREDICT algorithm)

Directly implements Algorithm 1 from Cohen et al. 2019.

Two-stage process:
1. **Cheap pass (N₀ samples):** identify the top class candidate ĉ_A
2. **Hypothesis test:** if ĉ_A's vote count is significantly above 50% (using a
   binomial test at significance α), return ĉ_A. Otherwise **abstain** (return -1).

The abstention is honest — it means "I'm not confident enough to make a
certified claim." In practice, abstention rate is low for a well-trained model.

In [ ]:
def predict(image: Image.Image, sigma: float = SIGMA,
            n0: int = N0, alpha: float = ALPHA) -> int:
    """
    PREDICT: returns the smoothed classifier's prediction, or ABSTAIN (-1).

    Uses a two-stage procedure:
      1. N₀ cheap samples to identify the candidate top class ĉ_A
      2. Hypothesis test: if ĉ_A is the strict majority with confidence 1-α,
         return ĉ_A. Otherwise abstain.

    Args:
        image : PIL image (RGB)
        sigma : Gaussian noise level
        n0    : samples for the identification pass
        alpha : failure probability for the hypothesis test

    Returns:
        int — predicted class index, or ABSTAIN (-1)
    """
    x = _pil_to_pixels(image)

    # Stage 1: identify candidate class with N₀ samples
    counts0 = _predict_noisy_batch(x, n0, sigma)
    c_A = int(counts0.argmax())   # candidate top class

    # Stage 2: test whether c_A is the statistical majority
    # We get N more samples and count how many predict c_A
    counts1 = _predict_noisy_batch(x, N, sigma)
    k_A = int(counts1[c_A])       # votes for c_A out of N

    # One-sided binomial test: is k_A significantly > N/2?
    # p_value = P(Binomial(N, 0.5) >= k_A) — if small, c_A is the true majority
    p_value = stats.binom_test(k_A, N, 0.5, alternative='greater')

    if p_value <= alpha:
        return c_A
    else:
        return ABSTAIN   # not confident enough — abstain


print('predict() defined.')
print(f'ABSTAIN sentinel : {ABSTAIN}')

---
## 5. Certified Radius Computation (CERTIFY algorithm)

Directly implements Algorithm 2 from Cohen et al. 2019.

Given input x, the certified radius R is:

$$R = \sigma \cdot \Phi^{-1}(\underline{p}_A)$$

Where:
- σ is the noise level
- Φ⁻¹ is the inverse Gaussian CDF (probit function)
- p_A is a **lower confidence bound** on P[f(x + ε) = ĉ_A]

The lower bound p_A is computed via the Clopper-Pearson method:
```python
proportion_confint(k_A, N, alpha=2*alpha, method='beta')[0]
```
This gives a one-sided (1-α) lower bound on the true proportion,
meaning p_A ≤ true probability with probability at most α.

The certificate then says: **no L₂ attack of norm < R can change the prediction**
with probability at least 1-α over the randomness in the smoothing.

In [ ]:
def certify(image: Image.Image, sigma: float = SIGMA,
            n0: int = N0, n: int = N, alpha: float = ALPHA) -> tuple:
    """
    CERTIFY: returns the smoothed prediction and its certified L₂ radius.

    Implements Algorithm 2 from Cohen et al. (ICML 2019).

    Args:
        image : PIL image (RGB)
        sigma : Gaussian noise standard deviation
        n0    : samples for identifying the top class (cheap pass)
        n     : samples for computing the certified radius (expensive pass)
        alpha : failure probability — certificate holds with prob ≥ 1 - alpha

    Returns:
        (prediction, radius)
          - prediction : int class index, or ABSTAIN (-1)
          - radius     : float certified L₂ radius, or 0.0 if abstaining
    """
    x = _pil_to_pixels(image)

    # ── Stage 1: identify top class candidate with N₀ samples ─────────────────
    counts0 = _predict_noisy_batch(x, n0, sigma)
    c_A     = int(counts0.argmax())   # candidate top class

    # ── Stage 2: estimate p_A with N samples ──────────────────────────────────
    counts1 = _predict_noisy_batch(x, n, sigma)
    k_A     = int(counts1[c_A])       # votes for c_A out of N

    # ── Clopper-Pearson lower bound on p_A ────────────────────────────────────
    # proportion_confint returns (lower, upper); we want the lower bound
    # alpha=2*alpha because this is a one-sided bound (Cohen et al. Appendix C)
    p_A_lower = proportion_confint(k_A, n, alpha=2 * alpha, method='beta')[0]

    # ── Certified radius formula ───────────────────────────────────────────────
    # R = sigma * Phi_inverse(p_A)
    # Only certifiable if p_A > 0.5 (top class is the true majority with confidence)
    if p_A_lower > 0.5:
        radius = float(sigma * scipy_norm.ppf(p_A_lower))
        return c_A, radius
    else:
        # Abstain: cannot certify — p_A is not significantly above 0.5
        return ABSTAIN, 0.0


print('certify() defined.')
print()
print('Formula: R = σ · Φ⁻¹(p_A)')
print(f'  σ = {SIGMA}  (noise level)')
print(f'  Φ⁻¹ = scipy.stats.norm.ppf  (inverse Gaussian CDF)')
print(f'  p_A = Clopper-Pearson lower bound on top-class vote probability')

### 5.1 Sanity check on a single image

In [ ]:
# ── Pick any test image ───────────────────────────────────────────────────────
SAMPLE_IMAGE_PATH = IMAGES_DIR / '00000001.jpg'

if SAMPLE_IMAGE_PATH.exists():
    t0 = time.time()
    img = Image.open(SAMPLE_IMAGE_PATH).convert('RGB')

    pred, radius = certify(img)

    elapsed = time.time() - t0
    print('Certification result on single image:')
    print(f'  Prediction      : {IDX_TO_LABEL.get(pred, "ABSTAIN") if pred != ABSTAIN else "ABSTAIN"}')
    print(f'  Certified radius: {radius:.4f}  (L₂ norm)')
    print(f'  Certified radius: {radius:.4f}  means no L₂ attack < {radius:.4f} can flip this prediction')
    print(f'  Time taken      : {elapsed:.1f}s  (N={N} samples)')
    print()
    if pred != ABSTAIN:
        print(f'✓ Certificate holds with probability ≥ {1-ALPHA:.1%}')
    else:
        print('⚠ Abstained — smoothed classifier is not confident enough to certify.')
        print(f'  Try reducing SIGMA (current: {SIGMA}) or increasing N (current: {N}).')
else:
    print(f'⚠ Image not found: {SAMPLE_IMAGE_PATH}')
    print('Update SAMPLE_IMAGE_PATH to any .jpg in your MIO-TCD train/ folder.')

---
## 6. Evaluate on Test Set

For each test image, run CERTIFY and collect:
- Whether the smoothed prediction is correct
- The certified radius (or 0 if abstained)

This gives us the **certified accuracy at radius R** curve in Section 7.

In [ ]:
class MIOTCDDataset(torch.utils.data.Dataset):
    """
    Reads labelled_manifest.csv from Walid's pipeline.
    Identical to the class in trafficguard_model_v1.ipynb.
    """
    def __init__(self, manifest_path, split, images_dir=None):
        df = pd.read_csv(manifest_path, dtype={'image_id': str})
        self.df         = df[df['split'] == split].reset_index(drop=True)
        self.images_dir = Path(images_dir) if images_dir else None
        self.df = self.df.copy()
        self.df['label_idx'] = self.df['congestion_label'].map(LABEL_MAP)

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, row):
        if self.images_dir:
            return self.images_dir / f"{row['image_id']}.jpg"
        return Path(row['image_path'])

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = self._resolve_path(row)
        image = Image.open(path).convert('RGB')
        label = int(row['label_idx'])
        return image, label


print('MIOTCDDataset defined.')

In [ ]:
if MANIFEST_PATH.exists():
    test_dataset = MIOTCDDataset(MANIFEST_PATH, 'test', images_dir=IMAGES_DIR)
    test_dataset.df = test_dataset.df.sample(
        min(N_EVAL, len(test_dataset)), random_state=42
    ).reset_index(drop=True)
    print(f'Test set ready: {len(test_dataset)} images')
else:
    print(f'⚠ Manifest not found at {MANIFEST_PATH}')
    print('Running in single-image mode — update MANIFEST_PATH to evaluate on the full test set.')
    test_dataset = None

In [ ]:
if test_dataset is not None:
    results = []

    print(f'Certifying {len(test_dataset)} images (σ={SIGMA}, N={N})...')
    print(f'Estimated time: {len(test_dataset) * N / 1000 * 3:.0f}–{len(test_dataset) * N / 1000 * 5:.0f}s on GPU')
    print()

    for i in range(len(test_dataset)):
        img, true_label = test_dataset[i]
        t0              = time.time()

        pred, radius = certify(img, sigma=SIGMA, n0=N0, n=N, alpha=ALPHA)

        elapsed = time.time() - t0
        correct = (pred == true_label) if pred != ABSTAIN else False
        abstained = (pred == ABSTAIN)

        results.append({
            'true_label': true_label,
            'pred':       pred,
            'radius':     radius,
            'correct':    correct,
            'abstained':  abstained,
            'time_s':     round(elapsed, 2),
        })

        if (i + 1) % 10 == 0 or i == 0:
            done_correct  = sum(r['correct']   for r in results)
            done_abstain  = sum(r['abstained'] for r in results)
            done_n        = len(results)
            avg_rad       = np.mean([r['radius'] for r in results if not r['abstained']] or [0])
            print(f'  {done_n:>3}/{len(test_dataset)}  '
                  f'correct: {done_correct/done_n:.0%}  '
                  f'abstained: {done_abstain/done_n:.0%}  '
                  f'avg_radius: {avg_rad:.3f}')

    results_df = pd.DataFrame(results)
    print(f'\nDone.')
    print(f'  Certified correct  : {results_df["correct"].sum()} / {len(results_df)} ({results_df["correct"].mean():.1%})')
    print(f'  Abstained          : {results_df["abstained"].sum()} / {len(results_df)} ({results_df["abstained"].mean():.1%})')
    print(f'  Avg certified radius (non-abstained): {results_df[~results_df["abstained"]]["radius"].mean():.4f}')
    results_df.to_csv('randomised_smoothing_results.csv', index=False)
    print(f'  Saved: randomised_smoothing_results.csv')

---
## 7. Certified Accuracy Curve

The certified accuracy at radius R = fraction of test images that are BOTH:
1. Correctly classified by the smoothed classifier g
2. Have certified radius ≥ R (i.e. provably robust against any L₂ attack < R)

This is the canonical plot from Cohen et al. 2019 — the standard way to
communicate randomised smoothing results in papers and reports.

In [ ]:
if test_dataset is not None and len(results_df) > 0:
    radii = np.linspace(0.0, 1.5, 300)

    # Certified accuracy at each radius:
    # fraction of images that are correct AND have certified_radius >= r
    cert_acc = [
        (results_df['correct'] & (results_df['radius'] >= r)).mean()
        for r in radii
    ]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(radii, cert_acc, color='#2DD4BF', linewidth=2.5,
            label=f'Certified accuracy (σ={SIGMA})')
    ax.axhline(results_df['correct'].mean(), color='#F0524B',
               linestyle='--', linewidth=1.5,
               label=f'Clean accuracy of smoothed classifier ({results_df["correct"].mean():.1%})')

    ax.set_xlabel('L₂ certified radius R', fontsize=12)
    ax.set_ylabel('Certified accuracy', fontsize=12)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlim(0, 1.5)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_title(
        f'Randomised Smoothing — Certified Accuracy Curve\n'
        f'σ={SIGMA}  |  N={N} samples  |  α={ALPHA}  |  {len(results_df)} test images',
        fontsize=12
    )

    plt.tight_layout()
    plt.savefig('certified_accuracy_curve.png', dpi=120)
    plt.show()
    print('Saved: certified_accuracy_curve.png')

---
## 8. Sigma Sensitivity Analysis

σ controls the fundamental tradeoff in randomised smoothing:

- **Higher σ** → more noise → larger certified radius → lower clean accuracy
- **Lower σ** → less noise → smaller certified radius → higher clean accuracy

Standard values from the literature: 0.12, 0.25, 0.50, 1.00.
We sweep a subset here to find the sweet spot for our traffic classification task.

Note: this cell runs the full certification loop for each σ value,
so it is slow. Run it after confirming Section 6 works end-to-end.

In [ ]:
SIGMA_VALUES = [0.12, 0.25, 0.50]
N_SIGMA_EVAL = min(30, N_EVAL)   # use fewer images for the sweep

if test_dataset is not None:
    sigma_summary = []
    print(f'Sigma sweep on {N_SIGMA_EVAL} images...')

    for sig in SIGMA_VALUES:
        sig_results = []
        for i in range(N_SIGMA_EVAL):
            img, true_label = test_dataset[i]
            pred, radius    = certify(img, sigma=sig, n0=N0, n=N, alpha=ALPHA)
            correct   = (pred == true_label) if pred != ABSTAIN else False
            abstained = (pred == ABSTAIN)
            sig_results.append({'correct': correct, 'abstained': abstained, 'radius': radius})

        sig_df = pd.DataFrame(sig_results)
        clean_acc  = sig_df['correct'].mean()
        abstain_rate = sig_df['abstained'].mean()
        avg_radius = sig_df[~sig_df['abstained']]['radius'].mean() if not sig_df['abstained'].all() else 0.0

        sigma_summary.append({
            'sigma':        sig,
            'clean_acc':    round(clean_acc, 4),
            'abstain_rate': round(abstain_rate, 4),
            'avg_radius':   round(avg_radius, 4),
        })
        print(f'  σ={sig:.2f} → clean_acc={clean_acc:.1%}  '
              f'abstain={abstain_rate:.1%}  avg_radius={avg_radius:.4f}')

    sigma_df = pd.DataFrame(sigma_summary)
    print()
    print(sigma_df.to_string(index=False))

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar([str(s) for s in SIGMA_VALUES], sigma_df['clean_acc'],
                color='#2DD4BF', edgecolor='white', width=0.5)
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    axes[0].set_title('Clean accuracy of smoothed classifier vs σ')
    axes[0].set_xlabel('σ (noise level)')
    axes[0].set_ylabel('Accuracy')
    axes[0].grid(axis='y', alpha=0.3)

    axes[1].bar([str(s) for s in SIGMA_VALUES], sigma_df['avg_radius'],
                color='#F5A623', edgecolor='white', width=0.5)
    axes[1].set_title('Average certified radius vs σ')
    axes[1].set_xlabel('σ (noise level)')
    axes[1].set_ylabel('Avg certified L₂ radius')
    axes[1].grid(axis='y', alpha=0.3)

    fig.suptitle('Sigma sensitivity — accuracy vs certified radius tradeoff', fontsize=12)
    plt.tight_layout()
    plt.savefig('sigma_sensitivity.png', dpi=120)
    plt.show()
    print('Saved: sigma_sensitivity.png')

---
## 9. Visualise — Certified vs Uncertified Examples

Show a grid of test images labelled by:
- Prediction (correct / wrong)
- Certified radius
- Whether the classifier abstained

This makes the "certified" claim concrete and visual for the demo.

In [ ]:
if test_dataset is not None and len(results_df) > 0:
    CLASS_COLORS = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}

    # Separate certified-correct from uncertified/wrong
    cert_correct  = results_df[results_df['correct'] & ~results_df['abstained']].head(4)
    not_certified = results_df[results_df['abstained'] | ~results_df['correct']].head(4)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(
        f'Randomised Smoothing (σ={SIGMA}) — Certified vs Not Certified\n'
        f'Top row: certified correct  |  Bottom row: abstained or wrong',
        fontsize=12
    )

    def show_row(row_axes, subset, row_label):
        for ax, (_, row) in zip(row_axes, subset.iterrows()):
            idx = subset.index.get_loc(row.name)
            img, _ = test_dataset[idx]
            ax.imshow(img.resize((224, 224)))
            ax.axis('off')
            pred_name = IDX_TO_LABEL.get(int(row['pred']), 'ABSTAIN') if row['pred'] != ABSTAIN else 'ABSTAIN'
            true_name = IDX_TO_LABEL[int(row['true_label'])]
            color     = CLASS_COLORS.get(pred_name, 'gray')
            ax.set_title(
                f'True: {true_name}\nPred: {pred_name}\nR = {row["radius"]:.3f}',
                fontsize=9, color=color, fontweight='bold'
            )
        # Fill any empty slots in the row
        for ax in row_axes[len(subset):]:
            ax.axis('off')

    show_row(axes[0], cert_correct,  'Certified correct')
    show_row(axes[1], not_certified, 'Not certified / wrong')

    plt.tight_layout()
    plt.savefig('certified_examples.png', dpi=120)
    plt.show()
    print('Saved: certified_examples.png')

---
## 10. Summary

In [ ]:
if test_dataset is not None and len(results_df) > 0:
    cert_at_025 = (results_df['correct'] & (results_df['radius'] >= 0.25)).mean()
    cert_at_050 = (results_df['correct'] & (results_df['radius'] >= 0.50)).mean()

    print('TRAFFICGUARD — RANDOMISED SMOOTHING SUMMARY')
    print('=' * 58)
    print(f'Reference         : Cohen, Rosenfeld & Kolter, ICML 2019')
    print(f'Base classifier   : ResNet18 (Soham\'s best.pt)')
    print(f'Sigma (σ)         : {SIGMA}  (noise standard deviation)')
    print(f'N₀                : {N0}  (prediction samples)')
    print(f'N                 : {N}  (certification samples)')
    print(f'Alpha (α)         : {ALPHA}  (failure probability)')
    print(f'Confidence        : {(1-ALPHA)*100:.1f}%  (certificate holds with this probability)')
    print(f'Evaluated on      : {len(results_df)} test images')
    print()
    print('Results:')
    print(f'  Correct predictions      : {results_df["correct"].mean():.1%}')
    print(f'  Abstention rate          : {results_df["abstained"].mean():.1%}')
    non_abs = results_df[~results_df["abstained"]]
    if len(non_abs):
        print(f'  Avg certified radius     : {non_abs["radius"].mean():.4f}  (L₂ norm)')
        print(f'  Max certified radius     : {non_abs["radius"].max():.4f}')
    print(f'  Certified accuracy @ R=0.25: {cert_at_025:.1%}')
    print(f'  Certified accuracy @ R=0.50: {cert_at_050:.1%}')
    print()
    print('What these numbers mean:')
    print(f'  {cert_at_025:.1%} of test images are PROVABLY robust against any L₂ attack < 0.25')
    print(f'  {cert_at_050:.1%} of test images are PROVABLY robust against any L₂ attack < 0.50')
    print(f'  This is a mathematical guarantee, not just an empirical observation.')
    print()
    print('Saved outputs:')
    for f in ['randomised_smoothing_results.csv', 'certified_accuracy_curve.png',
              'sigma_sensitivity.png', 'certified_examples.png']:
        status = '✓' if Path(f).exists() else '✗ not yet generated'
        print(f'  {status}  {f}')
    print()
    print('Key design decisions:')
    print('  - N=1000 samples per image: balances tightness vs speed')
    print('  - Clopper-Pearson lower bound: conservative, statistically rigorous')
    print('  - Abstention: honest — reports uncertainty rather than a wrong certificate')
    print('  - Noise added in [0,1] pixel space before model normalisation (correct)')
    print()
    print('Limitations to note in final report:')
    print('  - Certificate is in L₂ norm, not L∞ (FGSM/PGD use L∞ by default)')
    print('  - Accuracy drops compared to the non-smoothed base classifier')
    print('  - Larger N gives tighter radii but is computationally expensive')
    print('  - CIFAR-10 trained DDPM used for diffusion comparison — RS uses ResNet18 directly')